In [4]:
import os
import sys
import numpy as np
notebook_dir = os.path.abspath(os.path.dirname(''))
project_root = os.path.dirname(notebook_dir)
sys.path.append(project_root)
import pandas as pd
from src.utils.db_utils import get_connection, execute_query
from src.features.team_level_features import defensive_metrics
from src.features.team_level_features import offensive_mettrics
from src.utils.get_opponent_utils import create_defense_team_id, fetch_and_get_opponents

season = 2024
stat = "total_yards"

# Test the function
"""
stats_pg_df, average_stats_pg_df = offensive_mettrics(season, stat)
stats_allowed_df, average_allowed_df = defensive_metrics(season, stat)


print("Yards Per Game:")
print(stats_pg_df)

print("\nAverage Yards :")
print(average_stats_pg_df)

print("\nAverage Yards Allowed:")
print(stats_allowed_df)
print("\nAverage Yards Allowed:")
print(average_allowed_df)
"""




'\nstats_pg_df, average_stats_pg_df = offensive_mettrics(season, stat)\nstats_allowed_df, average_allowed_df = defensive_metrics(season, stat)\n\n\nprint("Yards Per Game:")\nprint(stats_pg_df)\n\nprint("\nAverage Yards :")\nprint(average_stats_pg_df)\n\nprint("\nAverage Yards Allowed:")\nprint(stats_allowed_df)\nprint("\nAverage Yards Allowed:")\nprint(average_allowed_df)\n'

In [5]:
def defensive_metrics(query_results, stat):
    """
    Calculate defensive stats based on the opponent's offensive stats.

    Args:
        query_results (pd.DataFrame): DataFrame with team stats and defense IDs.
        stat (str): The statistic to calculate (e.g., "yards").

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Per-game defensive stats and rolling averages.
    """
    # Ensure the query_results DataFrame includes necessary columns
    if 'defenseid' not in query_results.columns:
        raise ValueError("The input DataFrame must include a 'defenseid' column.")

    # Extract unique weeks and teams
    weeks = query_results["week"].unique()
    teams = query_results["teamid"].unique()

    # Initialize DataFrame for defensive stats
    defensive_stats_pg_df = pd.DataFrame(index=weeks, columns=teams)

    # Calculate per-game defensive stats
    for week in weeks:
        current_week_data = query_results[query_results["week"] == week]
        for team in teams:
            # Find rows where the `defenseid` matches the team
            defense_rows = current_week_data[current_week_data["defenseid"] == team]
            if not defense_rows.empty:
                # Sum the `stat` column for the opposing team's offensive stats
                defensive_stats_pg_df.loc[week, team] = defense_rows[stat].sum()

    # Calculate rolling averages for defensive stats
    avg_defensive_stats_pg_df = pd.DataFrame(index=weeks, columns=teams)
    for week in weeks:
        for team in teams:
            avg_defensive_stats_pg_df.loc[week, team] = defensive_stats_pg_df.loc[:week, team].dropna().mean()

    return defensive_stats_pg_df, avg_defensive_stats_pg_df


In [6]:
season = 2024
stat = 'total_yards'

# Fetch data
query_results = fetch_and_get_opponents(season, stat)

# Inspect the results
print("Fetched Data with Defense ID:")
print(query_results.head())

defensive_stats_pg, avg_defensive_stats_pg = defensive_metrics(query_results, stat)

# Display the results
print("\nDefensive Stats Per Game:")
print(defensive_stats_pg)

print("\nAverage Defensive Stats Per Game:")
print(avg_defensive_stats_pg)

Successfully connected to the database!
Fetched Data with Defense ID:
                            teamstatsid    gamesummaryid teamid awayteamid  \
0  df6bae3b-0387-439a-a965-b854f923eebb  gs-202401BALKAN    BAL        BAL   
1  4dd877ea-00da-49d1-bcc4-96ddc82e4d06  gs-202401BALKAN    KAN        BAL   
2  3cee4249-eb58-441e-9de4-fd49b57e6beb  gs-202401GNBPHI    GNB        GNB   
3  7eea7b0e-6258-4a04-980b-6eb68b5f6a7a  gs-202401GNBPHI    PHI        GNB   
4  0dc27ef8-6e23-41f8-ac47-7349b1fe87a9  gs-202401PITATL    PIT        PIT   

  hometeamid  team time_of_possession  week  net_pass_yards  \
0        KAN  None              33:43     1             267   
1        KAN  None              26:17     1             281   
2        PHI  None              27:13     1             251   
3        PHI  None              32:47     1             266   
4        ATL  None              35:36     1             133   

              hometeam  ...  fumbles_lost turnovers         date sacked_yards  \
0